<a href="https://colab.research.google.com/github/rdelhibabu/Post-Quantum_Cryptographic_HQN/blob/main/Post_Quantum_Cryptographic_HQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Block 1: Setup & Configuration
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np
import matplotlib.pyplot as plt

# --- System & Cryptographic Parameters ---
# Hardware profile timings (Simulated ARM Cortex-M4 168 MHz)
CRYPTO_TIMINGS = {
    "ECDSA": {"sign_ms": 9.52, "verify_ms": 18.45, "size_bytes": 64},        #
    "ML-DSA": {"sign_ms": 49.40, "verify_ms": 14.58, "size_bytes": 2420},    #
    "Kyber": {"encaps_ms": 3.57, "decaps_ms": 3.27}                          #
}

# Network & FL Parameters
NUM_NODES = 100 # Scaled down from 10,000 for Colab execution time [cite: 341]
LOCAL_EPOCHS = 5 #
BATCH_SIZE = 64 #
GLOBAL_ROUNDS = 100 # [cite: 389]
ROTATION_ROUND = 40 #

# Threat Oracle States
class EpochState:
    CLASSICAL = 0     # Epoch 0 [cite: 307]
    PRE_COMMIT = 1    # Epoch 1 [cite: 307]
    HYBRID = 2        # Epoch 2 [cite: 307]
    POST_QUANTUM = 3  # Epoch 3 [cite: 307]

current_epoch_state = EpochState.CLASSICAL

In [2]:
# Block 2: Model & Network Logic
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ResNet-18 for CIFAR-10
model = torchvision.models.resnet18(num_classes=10)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model = model.to(device)

# CIFAR-10 Dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

def simulate_crypto_delay(state, num_nodes):
    """Simulates the computational latency injected by the cryptographic layer."""
    if state == EpochState.CLASSICAL:
        delay = (CRYPTO_TIMINGS["ECDSA"]["sign_ms"] + CRYPTO_TIMINGS["ECDSA"]["verify_ms"]) / 1000.0
    elif state == EpochState.PRE_COMMIT:
        # Simulating the 345ms transition latency spike
        delay = 0.345
    elif state == EpochState.HYBRID:
        # Dual-signature verification [cite: 309]
        delay = (CRYPTO_TIMINGS["ECDSA"]["sign_ms"] + CRYPTO_TIMINGS["ML-DSA"]["verify_ms"] * 2) / 1000.0
    else: # POST_QUANTUM
        delay = (CRYPTO_TIMINGS["ML-DSA"]["sign_ms"] + CRYPTO_TIMINGS["ML-DSA"]["verify_ms"]) / 1000.0

    # Simulate network processing time
    time.sleep(delay * (num_nodes / 100)) # Scaled for demonstration
    return delay

def evaluate_model(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

100%|██████████| 170M/170M [27:51<00:00, 102kB/s]


In [ ]:
# Block 3: Execution Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

accuracy_history = []
round_latencies = []

print("Starting Agile Federated Learning Simulation...")

for round_num in range(1, GLOBAL_ROUNDS + 1):
    start_time = time.time()

    # Threat Oracle Trigger Logic (Algorithm 3)
    if round_num == ROTATION_ROUND:
        print(f"\n--- [Oracle Triggered] Initiating PQC Migration at Round {round_num} ---")
        current_epoch_state = EpochState.PRE_COMMIT
    elif round_num == ROTATION_ROUND + 1:
        current_epoch_state = EpochState.HYBRID
    elif round_num == ROTATION_ROUND + 3:
        print("--- [Grace Period Ended] Hard Fork to ML-DSA ---")
        current_epoch_state = EpochState.POST_QUANTUM

    # Local Training Simulation (FedAvg)
    model.train()
    running_loss = 0.0
    # Simulate partial data pass for speed in Colab
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if i > 10: break # Truncated for simulation speed

    # Apply Cryptographic and Network Delays
    crypto_delay = simulate_crypto_delay(current_epoch_state, NUM_NODES)

    # Evaluation
    acc = evaluate_model(model, testloader)
    accuracy_history.append(acc)

    round_time = time.time() - start_time
    round_latencies.append(round_time)

    if round_num % 10 == 0 or current_epoch_state == EpochState.PRE_COMMIT:
        state_str = ["Classical", "Pre-Commit", "Hybrid", "Post-Quantum"][current_epoch_state]
        print(f"Round {round_num}/{GLOBAL_ROUNDS} | State: {state_str} | Acc: {acc:.2f}% | Latency: {round_time:.3f}s")

print("\nSimulation Complete. Peak Accuracy: {:.2f}%".format(max(accuracy_history)))

Starting Agile Federated Learning Simulation...


In [ ]:
# Block 4: Plotting Results
plt.figure(figsize=(10, 6))
plt.plot(range(1, GLOBAL_ROUNDS + 1), accuracy_history, marker='o', color='red', markersize=4, label='Agile Migration')

# Mark the rotation point [cite: 398]
plt.axvline(x=ROTATION_ROUND, color='black', linestyle='-.', label='Rotation Triggered (R=40)')

plt.title('Federated Learning Test Accuracy (CIFAR-10) During Cryptographic Rotation', fontsize=14)
plt.xlabel('Global Communication Round', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.grid(True)
plt.legend(loc='lower right')
plt.show()